# Exploracao do PlantVillage

Auditoria inicial do dataset PlantVillage usando apenas as imagens em `raw/color/`, seguida da divisao train/validation/test nos metadados, sem extrair ou copiar imagens e sem treinamento.

In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPOSITORY_URL = "https://github.com/murilodc/plant-disease-classification.git"
COLAB_PROJECT_DIR = Path("/content/plant-disease-classification")
LEGACY_COLAB_PROJECT_DIR = Path("/content/tcc-plant-disease-classification")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/plant-disease-classification")
LEGACY_DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/TCC")
AUTO_CLONE_REPOSITORY = True
MOUNT_DRIVE = True
if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")


def is_project_root(path: Path) -> bool:
    return (path / "src" / "plantvillage_audit.py").is_file()


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    if IN_COLAB:
        candidates.extend([
            DRIVE_PROJECT_DIR, LEGACY_DRIVE_PROJECT_DIR,
            COLAB_PROJECT_DIR, LEGACY_COLAB_PROJECT_DIR,
        ])
    for candidate in candidates:
        if is_project_root(candidate):
            return candidate

    if IN_COLAB and AUTO_CLONE_REPOSITORY:
        clone_target = DRIVE_PROJECT_DIR if MOUNT_DRIVE else COLAB_PROJECT_DIR
        if clone_target.exists():
            raise FileNotFoundError(
                f"{clone_target} já existe, mas não contém o projeto completo. "
                "Remova ou renomeie esse diretório e execute a célula novamente."
            )
        print("Clonando o repositório para o Colab...")
        subprocess.check_call([
            "git", "clone", "--depth", "1", REPOSITORY_URL, str(clone_target)
        ])
        if is_project_root(clone_target):
            return clone_target

    raise FileNotFoundError(
        "Projeto não encontrado. Envie o repositório completo, monte o Drive "
        "ou habilite AUTO_CLONE_REPOSITORY."
    )


BASE_DIR = find_project_root()

In [ ]:
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
SRC_DIR = BASE_DIR / "src"

ZIP_PATH = DATA_DIR / "data.zip"
LEAF_MAP_PATH = DATA_DIR / "leaf_grouping" / "leaf-map.json"
OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_raw_color.csv"
SPLIT_OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_split.csv"
COLOR_IMAGE_DIR = Path("/content/plantvillage_color") if IN_COLAB else DATA_DIR / "plantvillage_color"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Base:", BASE_DIR)
print("ZIP:", ZIP_PATH)
print("Leaf map:", LEAF_MAP_PATH)
print("Resultados:", RESULTS_DIR)
print("Imagens locais:", COLOR_IMAGE_DIR)

## Arquivos oficiais

In [ ]:
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download

In [ ]:
if not ZIP_PATH.exists():
    ZIP_PATH = Path(
        hf_hub_download(
            repo_id="mohanty/PlantVillage",
            filename="data.zip",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
    )

if not LEAF_MAP_PATH.exists():
    LEAF_MAP_PATH = Path(
        hf_hub_download(
            repo_id="mohanty/PlantVillage",
            filename="leaf_grouping/leaf-map.json",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
        )
    )

print("data.zip existe:", ZIP_PATH.exists())
print("leaf-map.json existe:", LEAF_MAP_PATH.exists())

if ZIP_PATH.exists():
    print(f"Tamanho do data.zip: {ZIP_PATH.stat().st_size / (1024 ** 3):.2f} GB")

## Metadados

In [ ]:
import importlib.util
import sys
import urllib.request

module_file = SRC_DIR / "plantvillage_audit.py"
required_markers = (
    "FALLBACK_PREFIX",
    "leaf_id_source",
    "origem_leaf_id",
    "numero_identificadores_agrupamento_unicos",
    "maiores_grupos_leaf_id",
    "maiores_grupos_fallback",
    "fallbacks_multiclasse",
    ".replace(\".JPG\", \"\")",
)

module_text = module_file.read_text(encoding="utf-8") if module_file.exists() else ""
if not all(marker in module_text for marker in required_markers):
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    module_url = "https://raw.githubusercontent.com/murilodc/plant-disease-classification/main/src/plantvillage_audit.py"
    module_text = urllib.request.urlopen(module_url).read().decode("utf-8")
    module_file.write_text(module_text, encoding="utf-8")
    print("Modulo atualizado em:", module_file)

module_text = module_file.read_text(encoding="utf-8")
missing_markers = [marker for marker in required_markers if marker not in module_text]
if missing_markers:
    raise RuntimeError(f"Modulo PlantVillage ainda esta desatualizado: {module_file}")

spec = importlib.util.spec_from_file_location("plantvillage_audit_runtime", module_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Nao foi possivel carregar o modulo: {module_file}")

pv_audit = importlib.util.module_from_spec(spec)
sys.modules["plantvillage_audit_runtime"] = pv_audit
spec.loader.exec_module(pv_audit)

audit_metadata = pv_audit.audit_metadata
build_metadata_dataframe = pv_audit.build_metadata_dataframe
save_metadata_csv = pv_audit.save_metadata_csv

print("Modulo carregado de arquivo:", pv_audit.__file__)

fallback_test = pv_audit.resolve_leaf_id("x.JPG", "Classe___Teste", {})
if fallback_test.get("leaf_id") != "fallback_x" or fallback_test.get("leaf_id_source") != "fallback":
    raise RuntimeError(f"Modulo PlantVillage desatualizado: {pv_audit.__file__}")

metadata = build_metadata_dataframe(
    zip_path=ZIP_PATH,
    leaf_map_path=LEAF_MAP_PATH,
)

print("Formato do DataFrame:", metadata.shape)
metadata.head()

## Resumo da auditoria

In [ ]:
auditoria = audit_metadata(metadata)

print("Chaves da auditoria:", list(auditoria))

auditoria["resumo"]

## Origem dos identificadores

In [ ]:
auditoria["origem_leaf_id"]

In [ ]:
associadas_leaf_map = metadata.loc[metadata["leaf_id_source"].eq("leaf-map")]
fallbacks = metadata.loc[metadata["leaf_id_source"].eq("fallback")]

print("Imagens associadas pelo leaf-map:", len(associadas_leaf_map))
print("Imagens usando fallback:", len(fallbacks))

## Status da associacao

In [ ]:
auditoria["status_leaf_id"]

## Maiores grupos leaf_id

In [ ]:
auditoria["maiores_grupos_leaf_id"]

## Maiores grupos fallback

In [ ]:
auditoria["maiores_grupos_fallback"]

## Colisoes de fallback

In [ ]:
resumo = auditoria["resumo"].iloc[0]

print("Fallbacks com mais de uma imagem:", resumo["fallbacks_com_mais_de_uma_imagem"])
print("Fallbacks em mais de uma classe:", resumo["fallbacks_em_mais_de_uma_classe"])

## Grupo fallback_r

In [ ]:
fallback_r = auditoria["fallback_r_imagens"]

print("Imagens no grupo fallback_r:", len(fallback_r))
if len(fallback_r) > 0:
    print("Classes no grupo fallback_r:", fallback_r["quantidade_classes_no_grupo"].iloc[0])
else:
    print("Classes no grupo fallback_r: 0")

fallback_r.head(20)

In [ ]:
auditoria["fallback_r_imagens_por_classe"]

## Fallbacks em mais de uma classe

In [ ]:
auditoria["fallbacks_multiclasse"]

## Imagens por classe

In [ ]:
auditoria["imagens_por_classe"]

## Salvar metadados brutos

In [ ]:
csv_path = save_metadata_csv(metadata, OUTPUT_CSV)

print("CSV salvo em:", csv_path)
print("Arquivo existe:", csv_path.exists())

## Divisao train/validation/test

In [ ]:
import importlib.util
import sys
from pathlib import Path

import pandas as pd

if "SRC_DIR" not in globals():
    current_dir = Path.cwd().resolve()
    BASE_DIR = current_dir.parent if current_dir.name == "notebooks" else current_dir
    SRC_DIR = BASE_DIR / "src"
    RESULTS_DIR = BASE_DIR / "results"
    SPLIT_OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_split.csv"

split_module_file = SRC_DIR / "plantvillage_split.py"
required_split_markers = (
    "SPLIT_SEED = 42",
    "split_metadata_by_leaf_id",
    "validate_leaf_id_single_class",
    "split_diagnostics",
)

split_module_text = split_module_file.read_text(encoding="utf-8") if split_module_file.exists() else ""
if not all(marker in split_module_text for marker in required_split_markers):
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    split_module_text = '"""Utilities for splitting PlantVillage metadata by leaf_id."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nSPLIT_SEED = 42\nSPLIT_PROPORTIONS = {\n    "train": 0.70,\n    "validation": 0.15,\n    "test": 0.15,\n}\nREQUIRED_COLUMNS = ["classe", "leaf_id"]\n\n\ndef split_metadata_by_leaf_id(\n    metadata: pd.DataFrame,\n    proportions: dict[str, float] | None = None,\n    seed: int = SPLIT_SEED,\n) -> pd.DataFrame:\n    """Return metadata with a split column, keeping each leaf_id indivisible."""\n    proportions = SPLIT_PROPORTIONS if proportions is None else proportions\n    _validate_proportions(proportions)\n    proportions = {split: float(proportions[split]) for split in SPLIT_PROPORTIONS}\n    split_names = list(SPLIT_PROPORTIONS)\n    validate_leaf_id_single_class(metadata)\n\n    groups = (\n        metadata.groupby(["classe", "leaf_id"], dropna=False)\n        .size()\n        .rename("quantidade_imagens")\n        .reset_index()\n    )\n\n    rng = np.random.default_rng(seed)\n    assignments: dict[object, str] = {}\n\n    for classe in sorted(groups["classe"].unique()):\n        class_groups = groups.loc[groups["classe"].eq(classe)].copy()\n        class_assignments = _split_class_groups(class_groups, proportions, split_names, rng)\n        assignments.update(class_assignments)\n\n    result = metadata.copy()\n    result["split"] = result["leaf_id"].map(assignments)\n\n    if result["split"].isna().any():\n        missing = int(result["split"].isna().sum())\n        raise RuntimeError(f"{missing} imagens ficaram sem split.")\n\n    return result\n\n\ndef validate_leaf_id_single_class(metadata: pd.DataFrame) -> None:\n    """Raise an error if any leaf_id appears in more than one class."""\n    _validate_required_columns(metadata, REQUIRED_COLUMNS)\n\n    missing = metadata[REQUIRED_COLUMNS].isna().any()\n    if missing.any():\n        columns = ", ".join(missing[missing].index)\n        raise ValueError(f"Colunas obrigatorias com valores ausentes: {columns}")\n\n    class_counts = metadata.groupby("leaf_id", dropna=False)["classe"].nunique(dropna=False)\n    invalid = class_counts.loc[class_counts > 1]\n    if invalid.empty:\n        return\n\n    examples = (\n        metadata.loc[metadata["leaf_id"].isin(invalid.index)]\n        .groupby("leaf_id", dropna=False)["classe"]\n        .apply(lambda values: ", ".join(sorted(values.astype(str).unique())))\n        .head(10)\n    )\n    details = "; ".join(f"{leaf_id}: {classes}" for leaf_id, classes in examples.items())\n    raise ValueError(\n        "Ha leaf_id associado a mais de uma classe. "\n        f"Total de leaf_id invalidos: {len(invalid)}. Exemplos: {details}"\n    )\n\n\ndef split_diagnostics(\n    metadata_split: pd.DataFrame,\n    expected_total: int = 54305,\n    expected_classes: int = 38,\n    split_names: tuple[str, ...] = ("train", "validation", "test"),\n) -> dict[str, pd.DataFrame]:\n    """Build validation tables for the metadata split."""\n    _validate_required_columns(metadata_split, [*REQUIRED_COLUMNS, "split"])\n\n    total_images = len(metadata_split)\n    split_series = metadata_split["split"].astype("string")\n\n    imagens_por_split = (\n        split_series.value_counts()\n        .reindex(split_names, fill_value=0)\n        .rename_axis("split")\n        .reset_index(name="quantidade_imagens")\n    )\n    imagens_por_split["percentual_imagens"] = (\n        imagens_por_split["quantidade_imagens"].div(total_images).mul(100).round(4)\n    )\n\n    leaf_split = metadata_split[["leaf_id", "split"]].drop_duplicates()\n    leaf_ids_por_split = (\n        leaf_split["split"]\n        .value_counts()\n        .reindex(split_names, fill_value=0)\n        .rename_axis("split")\n        .reset_index(name="quantidade_leaf_id")\n    )\n\n    imagens_por_classe_split = (\n        pd.crosstab(metadata_split["classe"], metadata_split["split"])\n        .reindex(columns=split_names, fill_value=0)\n        .sort_index()\n    )\n    percentual_classe_por_split = (\n        imagens_por_classe_split.div(imagens_por_classe_split.sum(axis=1), axis=0)\n        .mul(100)\n        .round(4)\n    )\n\n    classes_por_split = (\n        metadata_split.groupby("split")["classe"]\n        .nunique()\n        .reindex(split_names, fill_value=0)\n        .rename_axis("split")\n        .reset_index(name="quantidade_classes")\n    )\n\n    leaf_split_counts = metadata_split.groupby("leaf_id", dropna=False)["split"].nunique()\n    leaf_id_ok = bool(leaf_split_counts.le(1).all())\n    invalid_splits = ~split_series.isin(split_names)\n    assigned_ok = bool(not invalid_splits.any() and total_images == expected_total)\n    classes_ok = bool(\n        metadata_split["classe"].nunique() == expected_classes\n        and classes_por_split["quantidade_classes"].eq(expected_classes).all()\n        and imagens_por_classe_split.gt(0).all(axis=None)\n    )\n\n    validacoes = pd.DataFrame(\n        [\n            {\n                "validacao": "nenhum_leaf_id_em_mais_de_um_split",\n                "ok": leaf_id_ok,\n                "detalhe": f"{int(leaf_split_counts.gt(1).sum())} leaf_id repetidos",\n            },\n            {\n                "validacao": "todas_as_54305_imagens_atribuidas",\n                "ok": assigned_ok,\n                "detalhe": (\n                    f"{total_images} imagens; {int(split_series.isna().sum())} sem split; "\n                    f"{int(invalid_splits.sum())} splits invalidos"\n                ),\n            },\n            {\n                "validacao": "as_38_classes_aparecem_nos_tres_splits",\n                "ok": classes_ok,\n                "detalhe": (\n                    f"{metadata_split[\'classe\'].nunique()} classes no total; "\n                    f"minimo por split: {int(classes_por_split[\'quantidade_classes\'].min())}"\n                ),\n            },\n        ]\n    )\n\n    return {\n        "imagens_por_split": imagens_por_split,\n        "leaf_ids_por_split": leaf_ids_por_split,\n        "imagens_por_classe_split": imagens_por_classe_split,\n        "percentual_classe_por_split": percentual_classe_por_split,\n        "classes_por_split": classes_por_split,\n        "validacoes": validacoes,\n    }\n\n\ndef save_split_metadata_csv(metadata_split: pd.DataFrame, output_path: str | Path) -> Path:\n    """Save split metadata CSV and return its path."""\n    path = Path(output_path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    metadata_split.to_csv(path, index=False)\n    return path\n\n\ndef _split_class_groups(\n    class_groups: pd.DataFrame,\n    proportions: dict[str, float],\n    split_names: list[str],\n    rng: np.random.Generator,\n) -> dict[object, str]:\n    if len(class_groups) < len(split_names):\n        classe = class_groups["classe"].iloc[0]\n        raise ValueError(f"A classe {classe} nao possui leaf_id suficientes para tres splits.")\n\n    total = int(class_groups["quantidade_imagens"].sum())\n    targets = {split: total * proportions[split] for split in split_names}\n\n    groups = class_groups.assign(_random=rng.random(len(class_groups))).sort_values(\n        ["quantidade_imagens", "_random", "leaf_id"],\n        ascending=[False, True, True],\n    )\n\n    counts = {split: 0 for split in split_names}\n    leaf_counts = {split: 0 for split in split_names}\n    assignments: dict[object, str] = {}\n\n    for position, row in enumerate(groups.itertuples(index=False)):\n        leaf_id = row.leaf_id\n        weight = int(row.quantidade_imagens)\n        remaining = len(groups) - position - 1\n        empty_splits = [split for split in split_names if leaf_counts[split] == 0]\n        candidates = empty_splits if len(empty_splits) > remaining else split_names\n        split = min(\n            candidates,\n            key=lambda candidate: _score_after_move(counts, targets, candidate, weight),\n        )\n        assignments[leaf_id] = split\n        counts[split] += weight\n        leaf_counts[split] += 1\n\n    return _improve_assignments(groups, assignments, counts, leaf_counts, targets, split_names)\n\n\ndef _improve_assignments(\n    groups: pd.DataFrame,\n    assignments: dict[object, str],\n    counts: dict[str, int],\n    leaf_counts: dict[str, int],\n    targets: dict[str, float],\n    split_names: list[str],\n) -> dict[object, str]:\n    current_score = _score(counts, targets)\n\n    while True:\n        best_move = None\n        best_score = current_score\n\n        for row in groups.itertuples(index=False):\n            leaf_id = row.leaf_id\n            weight = int(row.quantidade_imagens)\n            origin = assignments[leaf_id]\n\n            if leaf_counts[origin] <= 1:\n                continue\n\n            for destination in split_names:\n                if destination == origin:\n                    continue\n\n                next_counts = counts.copy()\n                next_counts[origin] -= weight\n                next_counts[destination] += weight\n                next_score = _score(next_counts, targets)\n\n                if next_score < best_score:\n                    best_score = next_score\n                    best_move = (leaf_id, weight, origin, destination)\n\n        if best_move is None:\n            return assignments\n\n        leaf_id, weight, origin, destination = best_move\n        assignments[leaf_id] = destination\n        counts[origin] -= weight\n        counts[destination] += weight\n        leaf_counts[origin] -= 1\n        leaf_counts[destination] += 1\n        current_score = best_score\n\n\ndef _score_after_move(\n    counts: dict[str, int],\n    targets: dict[str, float],\n    split: str,\n    weight: int,\n) -> float:\n    next_counts = counts.copy()\n    next_counts[split] += weight\n    return _score(next_counts, targets)\n\n\ndef _score(counts: dict[str, int], targets: dict[str, float]) -> float:\n    total = sum(targets.values())\n    return sum(((counts[split] - target) / total) ** 2 for split, target in targets.items())\n\n\ndef _validate_proportions(proportions: dict[str, float]) -> None:\n    if set(proportions) != set(SPLIT_PROPORTIONS):\n        expected = ", ".join(SPLIT_PROPORTIONS)\n        raise ValueError(f"Splits esperados: {expected}")\n\n    total = sum(proportions.values())\n    if not np.isclose(total, 1.0):\n        raise ValueError(f"As proporcoes devem somar 1.0, mas somam {total}.")\n\n    if any(value <= 0 for value in proportions.values()):\n        raise ValueError("Todas as proporcoes devem ser positivas.")\n\n\ndef _validate_required_columns(metadata: pd.DataFrame, columns: list[str]) -> None:\n    missing = [column for column in columns if column not in metadata]\n    if missing:\n        raise ValueError(f"Colunas obrigatorias ausentes: {\', \'.join(missing)}")\n\n\n__all__ = [\n    "REQUIRED_COLUMNS",\n    "SPLIT_PROPORTIONS",\n    "SPLIT_SEED",\n    "save_split_metadata_csv",\n    "split_diagnostics",\n    "split_metadata_by_leaf_id",\n    "validate_leaf_id_single_class",\n]\n'
    split_module_file.write_text(split_module_text, encoding="utf-8")
    print("Modulo de split criado/atualizado em:", split_module_file)

spec = importlib.util.spec_from_file_location("plantvillage_split_runtime", split_module_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Nao foi possivel carregar o modulo: {{split_module_file}}")

pv_split = importlib.util.module_from_spec(spec)
sys.modules["plantvillage_split_runtime"] = pv_split
spec.loader.exec_module(pv_split)

SPLIT_SEED = pv_split.SPLIT_SEED
save_split_metadata_csv = pv_split.save_split_metadata_csv
split_diagnostics = pv_split.split_diagnostics
split_metadata_by_leaf_id = pv_split.split_metadata_by_leaf_id
validate_leaf_id_single_class = pv_split.validate_leaf_id_single_class

print("Modulo de split carregado de arquivo:", pv_split.__file__)

if SPLIT_OUTPUT_CSV.exists():
    metadata_split = pd.read_csv(SPLIT_OUTPUT_CSV)
    validacoes_split = split_diagnostics(metadata_split)
    split_csv_path = SPLIT_OUTPUT_CSV
    print("CSV com split existente carregado:", split_csv_path)
else:
    validate_leaf_id_single_class(metadata)
    metadata_split = split_metadata_by_leaf_id(metadata, seed=SPLIT_SEED)
    validacoes_split = split_diagnostics(metadata_split)

    if not validacoes_split["validacoes"]["ok"].all():
        raise RuntimeError("A divisao gerada nao passou em todas as validacoes.")

    split_csv_path = save_split_metadata_csv(metadata_split, SPLIT_OUTPUT_CSV)
    print("CSV com split salvo em:", split_csv_path)

if not validacoes_split["validacoes"]["ok"].all():
    raise RuntimeError("O CSV de split nao passou em todas as validacoes.")

print("Seed:", SPLIT_SEED)
print("Arquivo existe:", split_csv_path.exists())

metadata_split.head()

## Quantidade e percentual de imagens por split

In [ ]:
validacoes_split["imagens_por_split"]

## Quantidade de leaf_id por split

In [ ]:
validacoes_split["leaf_ids_por_split"]

## Quantidade de imagens de cada classe em cada split

In [ ]:
validacoes_split["imagens_por_classe_split"]

## Percentual de cada classe destinado a cada split

In [ ]:
validacoes_split["percentual_classe_por_split"]

## Classes por split

In [ ]:
validacoes_split["classes_por_split"]

## Confirmacoes da divisao

In [ ]:
validacoes_split["validacoes"]

## Preparacao PyTorch

In [ ]:
try:
    import torch
    import torchvision
except ImportError:
    %pip install -q torch torchvision
    import torch
    import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

In [ ]:
import importlib.util
import sys
from pathlib import Path

import pandas as pd

if "IN_COLAB" not in globals():
    IN_COLAB = "google.colab" in sys.modules

if "BASE_DIR" not in globals():
    current_dir = Path.cwd().resolve()
    BASE_DIR = current_dir.parent if current_dir.name == "notebooks" else current_dir
    DATA_DIR = BASE_DIR / "data"
    RESULTS_DIR = BASE_DIR / "results"
    SRC_DIR = BASE_DIR / "src"
    ZIP_PATH = DATA_DIR / "data.zip"
    SPLIT_OUTPUT_CSV = RESULTS_DIR / "plantvillage_metadata_split.csv"

if "COLOR_IMAGE_DIR" not in globals():
    COLOR_IMAGE_DIR = Path("/content/plantvillage_color") if IN_COLAB else DATA_DIR / "plantvillage_color"

pytorch_module_file = SRC_DIR / "plantvillage_pytorch.py"
required_pytorch_markers = (
    "class PlantVillageDataset",
    "create_dataloaders",
    "extract_raw_color_from_zip",
    "RandomRotation(15)",
    "ColorJitter(brightness=0.1, contrast=0.1)",
)

pytorch_module_text = pytorch_module_file.read_text(encoding="utf-8") if pytorch_module_file.exists() else ""
if not all(marker in pytorch_module_text for marker in required_pytorch_markers):
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    pytorch_module_text = '"""PyTorch data pipeline for PlantVillage metadata splits."""\n\nfrom __future__ import annotations\n\nimport random\nimport shutil\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\n\n\nIMAGE_SIZE = 224\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\nSPLITS = ("train", "validation", "test")\nDEFAULT_IMAGE_ROOT = Path("/content/plantvillage_color")\nIMAGE_EXTENSIONS = frozenset({".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"})\n\n\ndef extract_raw_color_from_zip(\n    zip_path: str | Path,\n    output_dir: str | Path = DEFAULT_IMAGE_ROOT,\n    overwrite: bool = False,\n) -> dict[str, int | str]:\n    """Extract only raw/color images to one local directory."""\n    zip_path = Path(zip_path)\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    output_root = output_dir.resolve()\n\n    extracted = 0\n    skipped = 0\n\n    with zipfile.ZipFile(zip_path, "r") as zip_file:\n        for member in zip_file.infolist():\n            if member.is_dir():\n                continue\n\n            try:\n                relative_parts = _raw_color_relative_parts(member.filename)\n            except ValueError:\n                continue\n\n            if PurePosixPath(relative_parts[-1]).suffix.lower() not in IMAGE_EXTENSIONS:\n                continue\n\n            destination = output_dir.joinpath(*relative_parts)\n            _validate_inside_root(destination, output_root)\n\n            if (\n                not overwrite\n                and destination.exists()\n                and destination.stat().st_size == member.file_size\n            ):\n                skipped += 1\n                continue\n\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            with zip_file.open(member, "r") as source, destination.open("wb") as target:\n                shutil.copyfileobj(source, target)\n            extracted += 1\n\n    return {\n        "output_dir": str(output_dir),\n        "extracted": extracted,\n        "skipped": skipped,\n        "total": extracted + skipped,\n    }\n\n\nclass PlantVillageDataset(Dataset):\n    def __init__(\n        self,\n        metadata_csv: str | Path | pd.DataFrame,\n        image_root: str | Path,\n        split: str,\n        class_to_idx: dict[str, int] | None = None,\n        transform: transforms.Compose | None = None,\n    ) -> None:\n        if split not in SPLITS:\n            raise ValueError(f"Split invalido: {split}. Use um de {SPLITS}.")\n\n        metadata = load_metadata(metadata_csv)\n        _validate_metadata_columns(metadata)\n\n        self.split = split\n        self.image_root = Path(image_root)\n        self.class_to_idx = build_class_to_idx(metadata) if class_to_idx is None else class_to_idx\n        self.idx_to_class = {index: classe for classe, index in self.class_to_idx.items()}\n        self.classes = [self.idx_to_class[index] for index in sorted(self.idx_to_class)]\n        self.transform = build_image_transforms(split) if transform is None else transform\n\n        split_metadata = metadata.loc[metadata["split"].eq(split)].copy().reset_index(drop=True)\n        if split_metadata.empty:\n            raise ValueError(f"Nenhuma imagem encontrada para o split {split}.")\n\n        unknown_classes = sorted(set(split_metadata["classe"]) - set(self.class_to_idx))\n        if unknown_classes:\n            raise ValueError(f"Classes sem indice: {unknown_classes}")\n\n        self.metadata = split_metadata\n        self.labels = (\n            self.metadata["classe"].map(self.class_to_idx).astype("int64").to_numpy()\n        )\n        self.image_paths = [\n            self.image_root.joinpath(*_raw_color_relative_parts(zip_path))\n            for zip_path in self.metadata["zip_path"]\n        ]\n\n    def __len__(self) -> int:\n        return len(self.metadata)\n\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:\n        image_path = self.image_paths[index]\n        if not image_path.exists():\n            raise FileNotFoundError(\n                f"Imagem nao encontrada: {image_path}. "\n                "Execute extract_raw_color_from_zip antes de criar as epocas."\n            )\n\n        with Image.open(image_path) as image:\n            image = image.convert("RGB")\n            image_tensor = self.transform(image)\n\n        return image_tensor, int(self.labels[index])\n\n\ndef create_dataloaders(\n    metadata_csv: str | Path | pd.DataFrame,\n    image_root: str | Path,\n    batch_size: int = 32,\n    num_workers: int = 2,\n    seed: int = 42,\n    pin_memory: bool | None = None,\n) -> dict[str, DataLoader]:\n    """Create train, validation and test DataLoaders from metadata splits."""\n    metadata = load_metadata(metadata_csv)\n    class_to_idx = build_class_to_idx(metadata)\n    pin_memory = torch.cuda.is_available() if pin_memory is None else pin_memory\n\n    generator = torch.Generator()\n    generator.manual_seed(seed)\n\n    dataloaders = {}\n    for split in SPLITS:\n        dataset = PlantVillageDataset(\n            metadata_csv=metadata,\n            image_root=image_root,\n            split=split,\n            class_to_idx=class_to_idx,\n        )\n        dataloaders[split] = DataLoader(\n            dataset,\n            batch_size=batch_size,\n            shuffle=split == "train",\n            num_workers=num_workers,\n            pin_memory=pin_memory,\n            worker_init_fn=_seed_worker,\n            generator=generator,\n        )\n\n    return dataloaders\n\n\ndef build_image_transforms(split: str) -> transforms.Compose:\n    if split == "train":\n        return transforms.Compose(\n            [\n                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\n                transforms.RandomRotation(15),\n                transforms.RandomHorizontalFlip(),\n                transforms.ColorJitter(brightness=0.1, contrast=0.1),\n                transforms.ToTensor(),\n                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n            ]\n        )\n\n    if split in {"validation", "test"}:\n        return transforms.Compose(\n            [\n                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\n                transforms.ToTensor(),\n                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n            ]\n        )\n\n    raise ValueError(f"Split invalido: {split}. Use um de {SPLITS}.")\n\n\ndef build_class_to_idx(\n    metadata_csv: str | Path | pd.DataFrame,\n    expected_classes: int = 38,\n) -> dict[str, int]:\n    metadata = load_metadata(metadata_csv)\n    if "classe" not in metadata:\n        raise ValueError("Coluna obrigatoria ausente: classe")\n\n    classes = sorted(metadata["classe"].astype(str).unique())\n    if len(classes) != expected_classes:\n        raise ValueError(f"Esperadas {expected_classes} classes, encontradas {len(classes)}.")\n\n    return {classe: index for index, classe in enumerate(classes)}\n\n\ndef load_metadata(metadata_csv: str | Path | pd.DataFrame) -> pd.DataFrame:\n    if isinstance(metadata_csv, pd.DataFrame):\n        return metadata_csv.copy()\n\n    return pd.read_csv(metadata_csv)\n\n\ndef _seed_worker(worker_id: int) -> None:\n    worker_seed = torch.initial_seed() % 2**32\n    np.random.seed(worker_seed)\n    random.seed(worker_seed)\n\n\ndef _validate_metadata_columns(metadata: pd.DataFrame) -> None:\n    required_columns = ["zip_path", "classe", "split"]\n    missing = [column for column in required_columns if column not in metadata]\n    if missing:\n        raise ValueError(f"Colunas obrigatorias ausentes: {\', \'.join(missing)}")\n\n\ndef _raw_color_relative_parts(zip_path: str | Path) -> list[str]:\n    normalized = str(zip_path).replace("\\\\", "/").lstrip("/")\n    parts = [part for part in normalized.split("/") if part]\n    lower_parts = [part.lower() for part in parts]\n\n    for index in range(len(parts) - 2):\n        if lower_parts[index] == "raw" and lower_parts[index + 1] == "color":\n            relative_parts = parts[index + 2 :]\n            if len(relative_parts) < 2:\n                break\n            if any(part in {"..", "."} for part in relative_parts):\n                raise ValueError(f"Caminho inseguro no ZIP: {zip_path}")\n            return relative_parts\n\n    raise ValueError(f"Caminho fora de raw/color: {zip_path}")\n\n\ndef _validate_inside_root(path: Path, root: Path) -> None:\n    resolved = path.resolve()\n    try:\n        resolved.relative_to(root)\n    except ValueError as exc:\n        raise ValueError(f"Caminho inseguro para extracao: {path}") from exc\n\n\n__all__ = [\n    "DEFAULT_IMAGE_ROOT",\n    "IMAGE_SIZE",\n    "IMAGENET_MEAN",\n    "IMAGENET_STD",\n    "SPLITS",\n    "PlantVillageDataset",\n    "build_class_to_idx",\n    "build_image_transforms",\n    "create_dataloaders",\n    "extract_raw_color_from_zip",\n    "load_metadata",\n]\n'
    pytorch_module_file.write_text(pytorch_module_text, encoding="utf-8")
    print("Modulo PyTorch criado/atualizado em:", pytorch_module_file)

spec = importlib.util.spec_from_file_location("plantvillage_pytorch_runtime", pytorch_module_file)
if spec is None or spec.loader is None:
    raise ImportError(f"Nao foi possivel carregar o modulo: {pytorch_module_file}")

pv_torch = importlib.util.module_from_spec(spec)
sys.modules["plantvillage_pytorch_runtime"] = pv_torch
spec.loader.exec_module(pv_torch)

IMAGE_SIZE = pv_torch.IMAGE_SIZE
IMAGENET_MEAN = pv_torch.IMAGENET_MEAN
IMAGENET_STD = pv_torch.IMAGENET_STD
PlantVillageDataset = pv_torch.PlantVillageDataset
build_class_to_idx = pv_torch.build_class_to_idx
create_dataloaders = pv_torch.create_dataloaders
extract_raw_color_from_zip = pv_torch.extract_raw_color_from_zip

print("Modulo PyTorch carregado de arquivo:", pv_torch.__file__)

## Extrair raw/color para armazenamento local

In [ ]:
extraction_summary = extract_raw_color_from_zip(
    zip_path=ZIP_PATH,
    output_dir=COLOR_IMAGE_DIR,
    overwrite=False,
)

if int(extraction_summary['total']) != 54_305:
    raise RuntimeError("Extração incompleta: esperadas 54.305 imagens raw/color.")

extraction_summary

## Criar Datasets e DataLoaders

In [ ]:
metadata_split_pipeline = pd.read_csv(SPLIT_OUTPUT_CSV)
class_to_idx = build_class_to_idx(metadata_split_pipeline)
num_workers = 2 if IN_COLAB else 0

dataloaders = create_dataloaders(
    metadata_csv=metadata_split_pipeline,
    image_root=COLOR_IMAGE_DIR,
    batch_size=16,
    num_workers=num_workers,
    seed=42,
)
datasets = {split: loader.dataset for split, loader in dataloaders.items()}

print("Raiz das imagens:", COLOR_IMAGE_DIR)
print("Batch size:", dataloaders["train"].batch_size)
print("Workers:", num_workers)

## Quantidade de imagens em cada Dataset

In [ ]:
pd.DataFrame(
    [
        {"split": split, "quantidade_imagens": len(dataset)}
        for split, dataset in datasets.items()
    ]
)

## Quantidade de classes e mapeamento

In [ ]:
class_mapping = pd.DataFrame(
    sorted(class_to_idx.items(), key=lambda item: item[1]),
    columns=["classe", "indice"],
)

print("Quantidade de classes:", len(class_to_idx))
print("Menor indice:", class_mapping["indice"].min())
print("Maior indice:", class_mapping["indice"].max())

class_mapping

## Formato de um tensor

In [ ]:
sample_image, sample_label = datasets["train"][0]

print("Formato do tensor:", tuple(sample_image.shape))
print("Tipo do tensor:", sample_image.dtype)
print("Label:", sample_label, type(sample_label))

## Intervalo e tipo dos labels

In [ ]:
pd.DataFrame(
    [
        {
            "split": split,
            "menor_label": int(dataset.labels.min()),
            "maior_label": int(dataset.labels.max()),
            "tipo_labels": str(dataset.labels.dtype),
        }
        for split, dataset in datasets.items()
    ]
)

## Carregar um batch

In [ ]:
batch_images, batch_labels = next(iter(dataloaders["train"]))

print("Imagens:", tuple(batch_images.shape), batch_images.dtype)
print("Labels:", tuple(batch_labels.shape), batch_labels.dtype)
print("Menor label no batch:", int(batch_labels.min()))
print("Maior label no batch:", int(batch_labels.max()))

## Visualizar imagens de um batch

In [ ]:
import matplotlib.pyplot as plt

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
images_to_show = min(8, batch_images.size(0))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, image, label in zip(axes.flat, batch_images[:images_to_show], batch_labels[:images_to_show]):
    image = (image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)
    ax.imshow(image)
    ax.set_title(datasets["train"].idx_to_class[int(label)], fontsize=9)
    ax.axis("off")

for ax in axes.flat[images_to_show:]:
    ax.axis("off")

plt.tight_layout()